# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [2]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [3]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [4]:
df.info()
#dtypes = There is 1 decimal, 1 integer, and 4 strings (price should be read as numbers)
#Many non-null types (7,8?)
# number of rows = 8
# item, qty, and ts have 1 null value.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  8 non-null      int64  
 1   item      7 non-null      object 
 2   category  8 non-null      object 
 3   qty       7 non-null      float64
 4   price     8 non-null      object 
 5   ts        7 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [5]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df[df.duplicated(keep=False)].copy() # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)
print(removed)
print(clean)

[duplicates] dropped exact duplicate rows (1 row(s))
1
   order_id          item category  qty  price                   ts
0         1  Cheeseburger     Food  2.0  $7.50  2026-09-05T12:03:00
1         1  Cheeseburger     Food  2.0  $7.50  2026-09-05T12:03:00


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [13]:
floa = clean['price'] = (
    df['price'].str.replace('$',"",regex=False).str.strip().astype(float)
)


assert clean['price'].dtype == float
log('dtype', 'dropped white space and dollar signs', floa)

[dtype] dropped white space and dollar signs (0     7.5
1     7.5
2     7.5
3    12.0
4    24.0
5     6.0
6     6.0
7    12.0
Name: price, dtype: float64 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [15]:
clean['qty'] = pd.to_numeric(clean['qty'], errors = 'coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty']<0).sum()   # TODO: count of negative quantities
print(missing)
print(negative)

# TODO: apply your decision, then log both separately

0
0


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [16]:
print('before:', sorted(clean['category'].unique()))

clean['category'] = (clean['category'].str.strip().str.lower()
                     .str.replace('-', '', regex=False))
# TODO: CATEGORY_MAP = {...} for the judgment calls

print('after: ', sorted(clean['category'].unique()))

before: ['Food']
after:  ['food']


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [17]:
print('before:', sorted(clean['item'].unique()))

clean['item'] = (clean['item'].str.strip().str.lower()
                     .str.replace('-', '', regex=False))
# TODO: CATEGORY_MAP = {...} for the judgment calls

print('after: ', sorted(clean['item'].unique()))

before: ['Cheeseburger']
after:  ['cheeseburger']


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [ ]:
# TODO

### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [ ]:
# TODO: assertions

# TODO: clean['revenue'] = ...
# TODO: print rows, units, revenue, distinct categories

### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [ ]:
show_log()

**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [18]:
# Checkpoint
rows_after = 7
revenue_after = 76.50
biggest_decision = 'kept the negative amount as the refund amount'
revenue_other_way = 94.50

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 7
revenue: 76.5
decision that mattered: kept the negative amount as the refund amount
revenue the other way: 94.5
